## GGSN PROJ2, by Kacper Gawroński
This notebook focuses on testing and finetuning CNN-based models for detection, the models of my choosing are: YOLO v8, FasterRCNN and RetinaNet.
For the detection task I've chosen the D-Fire dataset, which has around 30k images. There are two classes that are need to be detected `smoke` and `fire`. This dataset consists of images from different sources - from cameras, from web and contributors. One objective is to test the models against each other, and try using different augmentations to enhance the model predictions.

In [ ]:
DATA_DIR = "data"
MODEL_DIR = "models"

In [ ]:
import json
import os
import zipfile
from config import DATA_DIR

def extract(zip_name):
    filepath = DATA_DIR / zip_name
    with zipfile.ZipFile(filepath) as zip_ref:
        zip_ref.extractall(DATA_DIR)
    os.remove(filepath)


def _create_filename_to_img_path(train: bool = True):
    if train:
        json_path = DATA_DIR / "train" / "train_mapping.json"
        img_dir = DATA_DIR / "train" / "images"
    else:
        json_path = DATA_DIR / "test" / "test_mapping.json"
        img_dir = DATA_DIR / "test" / "images"

    if json_path.exists():
        print(f"Mapping {json_path.name} already exists")
        return

    name_mapping = {}
    valid_extensions = ['.jpg', '.jpeg', '.png', '.tiff', '.bmp']

    summator = 0

    for file in img_dir.iterdir():

        if not file.is_file():
            continue

        if file.suffix not in valid_extensions:
            continue

        name_mapping[file.stem] = file.name
        summator += 1

    print(f"Found and mapped {summator} images")

    with open(json_path, 'w', encoding='utf-8') as json_file:
        json.dump(name_mapping, json_file, indent=4, ensure_ascii=False)

    print(f"Successfully mapped to file {json_path.name}")


def get_mappings(train: bool = True):
    if train:
        json_path = DATA_DIR / "train" / "train_mapping.json"
    else:
        json_path = DATA_DIR / "test" / "test_mapping.json"

    with open(json_path, 'r', encoding='utf-8') as json_file:
        return json.load(json_file)

if __name__ == "__main__":
    extract(zip_name="D-Fire.zip")
    _create_filename_to_img_path(True)
    _create_filename_to_img_path(False)

In [ ]:
from math import floor

import numpy as np
import torch
from torchmetrics import Metric

def yolo2matplotlib(bboxes: list[float], img_shape: tuple[int, int]) -> list[float]:
    """
    Converts the yolo bbox format to x, y, width and height for matplotlib plotting
    """
    img_x, img_y = img_shape

    width = floor(bboxes[2] * img_x)
    height = floor(bboxes[3] * img_y)

    left_x = floor(bboxes[0] * img_x - width/2)
    left_y = floor(bboxes[1] * img_y - height/2)

    return [left_x, left_y, width, height]

def yolo2torch(bboxes, img_shape: tuple[int, int]) -> torch.Tensor:
    if bboxes is None or len(bboxes) == 0:
        return torch.zeros((0, 4), dtype=torch.float32)

    bboxes = torch.tensor(bboxes, dtype=torch.float32)

    img_x, img_y = img_shape

    width = bboxes[:, 2] * img_x
    height = bboxes[:, 3] * img_y

    left_x = bboxes[:, 0] * img_x - width / 2
    left_y = bboxes[:, 1] * img_y - height / 2
    right_x = bboxes[:, 0] * img_x + width / 2
    right_y = bboxes[:, 1] * img_y + height / 2

    converted_bboxes = torch.stack((left_x, left_y, right_x, right_y), dim=1)

    return converted_bboxes


def yolo2yolo(bboxes: list[float], img_shape: tuple[int, int]) -> list[float]:
    """
    Dummy function for training YOLO model on dataset
    """
    return bboxes

def bbox_area(bboxes: np.ndarray | torch.Tensor) -> np.ndarray | torch.Tensor:
    if bboxes is None or len(bboxes) == 0:
        return torch.zeros((0,), dtype=torch.float32)

    if isinstance(bboxes, np.ndarray):
        bboxes = torch.from_numpy(bboxes)

    area = (bboxes[:, 2] - bboxes[:, 0]) * (bboxes[:, 3] - bboxes[:, 1])
    return area

def collate_fn(batch):
    return tuple(zip(*batch))

def is_better_value(val1, val2, metric_used: Metric):
    """Returns whether val2 is better than val1 based on metric_used"""
    if metric_used.higher_is_better:
        return val1 < val2
    else:
        return val1 > val2

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from dataset import _gather_labels
from config import DATA_DIR
import random
from utils import yolo2matplotlib
from extract_data import get_mappings
from math import ceil

In [ ]:
def plot_random_samples(train: bool, how_many: int = 6):
    samples = _gather_labels(train)
    mapping = get_mappings(train)

    imgs_path = DATA_DIR / "train" / "images" if train else DATA_DIR / "test" / "images"
    indxs = random.sample(range(0, len(samples)), how_many)

    cols = min(3, how_many)
    rows = ceil(how_many / cols)

    fig, axs = plt.subplots(rows, cols, figsize=(6 * cols, 6 * rows))

    if rows == 1 and cols == 1:
        axs = np.array([[axs]])
    elif rows == 1 or cols == 1:
        axs = axs.reshape(rows, cols)

    for idx, img_idx in enumerate(indxs):
        row_idx = idx // cols
        col_idx = idx % cols

        curr_sample = samples[img_idx]
        img_path = imgs_path / mapping.get(curr_sample["filename"], "")

        if not img_path.exists():
            print(f"Brak pliku: {curr_sample['filename']}")
            continue

        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img_h, img_w, _ = img.shape

        axs[row_idx][col_idx].imshow(img)
        axs[row_idx][col_idx].axis('off')
        axs[row_idx][col_idx].grid(False)
        axs[row_idx][col_idx].set_title(f"{curr_sample["filename"]}")

        for label, bboxes in zip(curr_sample["labels"], curr_sample["bboxes"]):
            bbox_norm = yolo2matplotlib(bboxes, (img_w, img_h))
            label_name = "Fire" if label == 1 else "Smoke"
            color = "green" if label == 1 else "blue"

            rect_path = plt.Rectangle((bbox_norm[0], bbox_norm[1]), width=bbox_norm[2], height=bbox_norm[3],
                                      edgecolor=color, fill=False, linewidth=2, label=label_name)
            axs[row_idx][col_idx].text(x = bbox_norm[0] - 5, y = bbox_norm[1] - 5, s = label_name, color = "black", fontsize = "large")
            axs[row_idx][col_idx].add_patch(rect_path)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_random_samples(train=True, how_many=6)

In [ ]:
import os

import cv2
import torch
from albumentations import BboxParams
from matplotlib.transforms import Bbox
from torch.utils.data import Dataset
from torch import nn
import albumentations as A

from utils import yolo2yolo, yolo2matplotlib, yolo2torch, bbox_area
from extract_data import get_mappings
from torchmetrics import Metric

from config import DATA_DIR

In [ ]:

def _read_data_from_file(path):
    labels = []
    bboxes = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip().split()

            if len(line) != 5:
                raise ValueError("Expected 5 values for a label - class and bounding box coordinates")
            labels.append(int(line[0]))
            bboxes.append(list(map(float, line[1:])))
    return labels, bboxes


def _gather_labels(is_train):
    labels = {}

    if is_train:
        labels_path = DATA_DIR / "train" / "labels"
    else:
        labels_path = DATA_DIR / "test" / "labels"

    for idx, file in enumerate(labels_path.iterdir()):
        if not file.is_file():
            continue
        try:
            file_labels, file_bboxes = _read_data_from_file(file)
            labels[idx] = {
                "filename" : file.stem,
                "labels" : file_labels,
                "bboxes" : file_bboxes,
            }
        except ValueError as e:
            print(e)

    return labels


class FireDatasetForDetection(Dataset):
    def __init__(self, train=True, transforms: list | None = None, bbox_params: A.BboxParams | None = None, target_resolution = (512, 512)):

        if bbox_params is None:
            bbox_params = A.BboxParams(format = 'yolo', label_fields=['labels'], min_visibility=0.2, clip=True, filter_invalid_bboxes=True)

        if transforms is None or any([not isinstance(transforms, A.BasicTransform) for transforms in transforms]):
            print("Transforms can be only from albumentations library")
            self.transforms = [A.NoOp()]
        else:
            self.transforms = transforms

        self.target_res = target_resolution
        self.transforms = A.Compose([A.LongestMaxSize(max_size=target_resolution[0]),
            A.PadIfNeeded(
                min_height=target_resolution[0],
                min_width=target_resolution[1],
                border_mode=cv2.BORDER_CONSTANT
            )] + self.transforms + [A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0), A.ToTensorV2()], bbox_params=bbox_params, seed=42)

        self.data = _gather_labels(train)
        self.imgs_path = DATA_DIR / "train" / "images" if train else DATA_DIR / "test" / "images"
        self.file_mappings = get_mappings(train)


    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        datapoint_dict = self.data[idx]
        filename, labels, bboxes = datapoint_dict.values()
        image = cv2.imread(self.imgs_path / self.file_mappings[filename])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


        transform_dict = self.transforms(image=image, labels=labels, bboxes=bboxes)

        transformed_img = transform_dict["image"]
        transformed_labels = transform_dict["labels"]
        transformed_bboxes = transform_dict["bboxes"]


        transformed_bboxes = yolo2torch(transformed_bboxes, self.target_res)
        bboxes_areas = bbox_area(transformed_bboxes)
        transformed_labels = torch.tensor([label + 1 for label in transformed_labels], dtype=torch.int64)

        target = {
            "boxes": transformed_bboxes,
            "labels": transformed_labels,
            "image_id": torch.tensor([idx], dtype=torch.int64),
            "area": bboxes_areas,
            "iscrowd": torch.zeros(len(transformed_labels), dtype=torch.int64)
        }

        return transformed_img, target



class ModelHistory:
    """
    Simple class for tracking model's metrics, for each different evaluation add new step
    """

    def __init__(self, steps, name, *args):
        self.history = dict()
        self.name = name

        for step in steps:
            self.history[step] = {
               metric_name : [] for metric_name in args
            }

    def add_step_outcomes(self, step, **kwargs):
        for metric_name, metric_value in kwargs.items():
            if metric_name not in self.history[step]:
                self.history[step][metric_name] = []

            self.history[step][metric_name].append(metric_value)

    def return_targeted_metric_val(self, step: str, name: str):
        return self.history[step][name][-1]

In [ ]:
from functorch.dim import Tensor
from torch.utils.data import DataLoader
from torchmetrics import Metric
from tqdm import tqdm
import math
from dataset import FireDatasetForDetection, ModelHistory
import torch
from utils import collate_fn, is_better_value
from torchmetrics.detection.mean_ap import  MeanAveragePrecision
from config import MODEL_DIR
from torchvision.models.detection.faster_rcnn import fasterrcnn_resnet50_fpn, FastRCNNPredictor
from torchvision.models.detection.retinanet import retinanet_resnet50_fpn_v2, RetinaNetHead


def _train_one_epoch(model, train_data, device, optimizer):
    model.train()
    pbar = tqdm(train_data)

    for images, targets in pbar:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)

        losses = torch.stack(list(loss_dict.values())).sum()

        loss_value = losses.item()

        if not math.isfinite(loss_value):
            print(f"Loss is {loss_value}, stopping training")
            break

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()



def _val_one_epoch(model, val_data, device, history, metric: Metric):
    model.eval()
    pbar = tqdm(val_data)

    with torch.no_grad():
        for images, targets in pbar:
            images = list(image.to(device) for image in images)

            outputs = model(images)

            outputs = [{k: v.detach().cpu() for k, v in t.items()} for t in outputs]
            targets = [{k: v.cpu() for k, v in t.items()} for t in targets]
            metric.update(outputs, targets)

    outcomes = metric.compute()

    history.add_step_outcomes("val", **outcomes)



def train_model(model, batch_size, epochs, learning_rate, weight_decay, optimizer_class, device,
                metric = MeanAveragePrecision(), checkpoint_every: int = 5, target_metric = "map",
                target_res = (512, 512), num_workers = 0):

    optimizer = optimizer_class(model.parameters(), lr=learning_rate, weight_decay=weight_decay, )

    dataset_train = FireDatasetForDetection(train=True, target_resolution=target_res)
    dataset_val = FireDatasetForDetection(train=False, target_resolution=target_res)

    train_data = DataLoader(dataset_train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers = num_workers)
    val_data = DataLoader(dataset_val, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers = num_workers)

    history = ModelHistory(["train", "val"], model.__class__.__name__, metric)
    prev_metric_val = None

    for epoch in range(epochs):
        _train_one_epoch(model, train_data, device, optimizer)
        _val_one_epoch(model, val_data, device, history, metric)

        if epoch % checkpoint_every == 0:
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict()
            }
            torch.save(checkpoint, MODEL_DIR / f"model_{epoch}.pth")

        curr_metric_val = history.return_targeted_metric_val("val", target_metric)

        if prev_metric_val is None or \
        is_better_value(prev_metric_val, curr_metric_val, metric):

            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict()
            }

            torch.save(checkpoint, MODEL_DIR / f"best_model.pth")

            prev_metric_val = curr_metric_val
            print(f"Found new best model at epoch {epoch} with metric {target_metric} and value {curr_metric_val}")

        metric.reset()

    return model, history

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

resnet_faster = fasterrcnn_resnet50_fpn(weights='DEFAULT', min_size = 256, max_size = 256)

num_classes = 3
in_features = resnet_faster.roi_heads.box_predictor.cls_score.in_features
resnet_faster.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

resnet_faster.to(device)

batch_size = 16
epochs = 3
learning_rate = 0.001
weight_decay = 0.0005

trained_faster, faster_history = train_model(
    model=resnet_faster,
    batch_size=batch_size,
    epochs=epochs,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    optimizer_class=torch.optim.SGD,
    device=device,
    target_metric="map_50",
    checkpoint_every=5,
    target_res=(256, 256),
    num_workers=4
)

print(faster_history.history)

In [ ]:
retina_model = retinanet_resnet50_fpn_v2(weights='DEFAULT', min_size=256, max_size=256)

num_classes = 3

in_channels = retina_model.backbone.out_channels
num_anchors = retina_model.head.classification_head.num_anchors

retina_model.head = RetinaNetHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=num_classes
)

retina_model.to(device)

trained_retina, retina_history = train_model(
    model=retina_model,
    batch_size=batch_size,
    epochs=epochs,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    optimizer_class=torch.optim.SGD,
    device=device,
    target_metric="map_50",
    checkpoint_every=5,
    target_res=(256, 256),
    num_workers=4
)

print(retina_history.history)

In [ ]:
from ultralytics import YOLO
from config import DATA_DIR

def epoch_status(trainer):
    current_epoch = trainer.epoch
    current_map50 = trainer.metrics.get('metrics/mAP50(B)', 0.0)
    if hasattr(trainer, 'tloss'):
        train_loss = trainer.tloss.sum().item()
    else:
        train_loss = 0.0

    print(f"\n Epoch: {current_epoch} | Loss: {train_loss:.4f} | mAP50: {current_map50:.4f}")



In [ ]:
yolo_model = YOLO("yolov8n.pt")
yolo_model.add_callback('on_fit_epoch_end', epoch_status)
results = yolo_model.train(data="fire.yaml", epochs = 10, batch = 16, patience = 3, save=True, save_period = 2, imgsz = 512,
            project = str(DATA_DIR / "yolo"), name="test", workers = 4, cos_lr = True)


In [ ]:
def plot_history(step_name: str, model_history: ModelHistory, max_img_per_width = 3):
    import math
    try:
        metrics_used = model_history.history[step_name]
    except KeyError:
        print("No metrics found for step", step_name)
        return

    h = math.ceil(len(metrics_used) / max_img_per_width)

    fig, axs = plt.subplots(h, max_img_per_width, figsize=(24, h * 6))

    for idx, metric in enumerate(metrics_used):
        x_pos = idx // max_img_per_width
        y_pos = idx % max_img_per_width

        if h == 1:
            curr_ax = axs[y_pos]
        else:
            curr_ax: plt.Axes = axs[x_pos, y_pos]

        curr_ax.plot(metrics_used[metric])
        curr_ax.set_title(f"{metric} over {len(metrics_used[metric])} steps")
        curr_ax.set_xlabel("Step")
        curr_ax.set_ylabel(f"{metric} Value")

    plt.show()




In [ ]:
def _plot_step_for_models(names: list[str], data, ax: plt.Axes):
    import seaborn as sns
    for name, data in zip(names, data):
        sns.lineplot(x=range(len(data)), y=data, ax=ax, label=name)


def plot_histories(step_name: str, histories: list[ModelHistory], max_img_per_width = 3):
    import math
    try:
        metrics_used = histories[0].history[step_name]
    except KeyError:
        print("No metrics found for step", step_name)
        return

    h = math.ceil(len(metrics_used) / max_img_per_width)

    fig, axs = plt.subplots(h, max_img_per_width, figsize=(24, h * 6))

    models_names = [history.name for history in histories]

    for idx, metric in enumerate(metrics_used):
        x_pos = idx // max_img_per_width
        y_pos = idx % max_img_per_width

        if h == 1:
            curr_ax = axs[y_pos]
        else:
            curr_ax: plt.Axes = axs[x_pos, y_pos]

        _plot_step_for_models(models_names, [history.history[step_name][metric] for history in histories], curr_ax)

        curr_ax.set_title(f"{metric} over {len(metrics_used[metric])} steps")
        curr_ax.set_xlabel("Step")
        curr_ax.set_ylabel(f"{metric} Value")

    plt.show()